# 03 - Machine Learning Model Benchmarking & Evaluation

## Objective
Benchmark multiple predictive algorithms for clinical risk stratification:
1. **L2-Penalized Logistic Regression** (Interpretable linear baseline)
2. **Random Forest Classifier** (Ensemble bagging baseline)
3. **XGBoost (Extreme Gradient Boosting)** (Primary non-linear boosted tree model)

We evaluate discriminative power (ROC-AUC, PR-AUC), calibration (Brier score), sensitivity/recall for adverse events, and cross-cohort stability.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix, brier_score_loss, f1_score, accuracy_score
)

%matplotlib inline

## 1. Load Processed Train and Test Datasets

In [ ]:
from src.schema_mapping import PROCESSED_DIR

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

TARGET_COL = "target_risk"
SOURCE_COL = "source_dataset"

feature_cols = [c for c in train_df.columns if c not in [TARGET_COL, SOURCE_COL]]

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]
sources_test = test_df[SOURCE_COL]

print(f"Training set: {X_train.shape[0]:,} patients, {X_train.shape[1]} features")
print(f"Test set    : {X_test.shape[0]:,} patients, {X_test.shape[1]} features")

## 2. Load the Production XGBoost Pipeline
Let's inspect the serialized best pipeline trained via `src/train.py`.

In [ ]:
from src.preprocessing import ClinicalDataPreprocessor, EndToEndRiskPipeline

model_path = PROJECT_ROOT / "models" / "best_model.joblib"

if model_path.exists():
    best_pipeline = joblib.load(model_path)
    print(f"Loaded trained pipeline from {model_path}")
    print(f"Pipeline components: {best_pipeline.preprocessor.__class__.__name__} -> {best_pipeline.classifier.__class__.__name__}")
else:
    print(f"Model not found at {model_path}. Please train via python src/train.py")

## 3. Multi-Model Benchmarking on Test Cohort
Let's evaluate Logistic Regression, Random Forest, and XGBoost side-by-side.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Sample subset if test set is very large for quick demonstration
sample_idx = np.random.choice(len(X_train), size=min(15000, len(X_train)), replace=False)
X_train_sub = X_train.iloc[sample_idx]
y_train_sub = y_train.iloc[sample_idx]

prep = ClinicalDataPreprocessor()
X_tr_proc = prep.fit_transform(X_train_sub)
X_te_proc = prep.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost (Classifier)": XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, eval_metric="logloss", random_state=42)
}

results = {}
for name, clf in models.items():
    clf.fit(X_tr_proc, y_train_sub)
    y_pred_proba = clf.predict_proba(X_te_proc)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    auc = roc_auc_score(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    brier = brier_score_loss(y_test, y_pred_proba)
    
    results[name] = {
        "ROC-AUC": round(auc, 4),
        "PR-AUC": round(ap, 4),
        "F1-Score": round(f1, 4),
        "Accuracy": round(acc, 4),
        "Brier Score": round(brier, 4),
        "y_pred_proba": y_pred_proba
    }

bench_df = pd.DataFrame([{k: v for k, v in d.items() if k != "y_pred_proba"}], index=[name for name in results.keys()]).T
bench_df

## 4. Visualizing ROC Curves & Precision-Recall Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = {"Logistic Regression": "#3b82f6", "Random Forest": "#10b981", "XGBoost (Classifier)": "#ef4444"}

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_pred_proba"])
    ax1.plot(fpr, tpr, label=f"{name} (AUC = {res['ROC-AUC']:.3f})", color=colors[name], lw=2)
    
    prec, rec, _ = precision_recall_curve(y_test, res["y_pred_proba"])
    ax2.plot(rec, prec, label=f"{name} (PR-AUC = {res['PR-AUC']:.3f})", color=colors[name], lw=2)

ax1.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax1.set_title("Receiver Operating Characteristic (ROC)", fontsize=13, fontweight="bold")
ax1.set_xlabel("False Positive Rate (1 - Specificity)")
ax1.set_ylabel("True Positive Rate (Sensitivity)")
ax1.legend(loc="lower right")

ax2.set_title("Precision-Recall Curves", fontsize=13, fontweight="bold")
ax2.set_xlabel("Recall (Sensitivity)")
ax2.set_ylabel("Precision (Positive Predictive Value)")
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

## 5. Confusion Matrix & Clinical Decision Boundary (XGBoost)

In [ ]:
xgb_probs = results["XGBoost (Classifier)"]["y_pred_proba"]
cm = confusion_matrix(y_test, (xgb_probs >= 0.5).astype(int))

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(cm, cmap="Blues")
fig.colorbar(cax)

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12, fontweight="bold")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Low Risk (0)", "High Risk (1)"])
ax.set_yticklabels(["Low Risk (0)", "High Risk (1)"])
ax.set_xlabel("Predicted Clinical Label", fontweight="bold", labelpad=10)
ax.set_ylabel("Actual Ground Truth", fontweight="bold")
ax.set_title("XGBoost Confusion Matrix", fontweight="bold", pad=15)
plt.tight_layout()
plt.show()